# A little extra!

## New addition to Week 1

### The Unreasonable Effectiveness of the Agent Loop

# What is an Agent?

## Three competing definitions

1. AI systems that can do work for you independently - Sam Altman

2. A system in which an LLM controls the workflow - Anthropic

3. An LLM agent runs tools in a loop to achieve a goal

## The third one is the new, emerging definition

But what does it mean?

Let's make it real.

In [1]:
# Start with some imports - rich is a library for making formatted text output in the terminal

from rich.console import Console
from dotenv import load_dotenv
from openai import OpenAI
import json
load_dotenv(override=True)

False

In [2]:
def show(text):
    try:
        Console().print(text)
    except Exception:
        print(text)

In [3]:
openai = OpenAI(
    base_url="http://10.1.90.100:11434/v1",
    api_key="ollama"
)

In [4]:
# Some lists!

todos = []
completed = []

In [5]:
def get_todo_report() -> str:
    result = ""
    for index, todo in enumerate(todos):
        if completed[index]:
            result += f"Todo #{index + 1}: [green][strike]{todo}[/strike][/green]\n"
        else:
            result += f"Todo #{index + 1}: {todo}\n"
    show(result)
    return result

In [6]:
get_todo_report()

''

In [7]:
def create_todos(descriptions: list[str]) -> str:
    todos.extend(descriptions)
    completed.extend([False] * len(descriptions))
    return get_todo_report()

In [8]:
def mark_complete(index: int, completion_notes: str) -> str:
    if 1 <= index <= len(todos):
        completed[index - 1] = True
    else:
        return "No todo at this index."
    Console().print(completion_notes)
    return get_todo_report()

In [9]:
todos, completed = [], []

create_todos(["Buy groceries", "Finish extra lab", "Eat banana"])

Todo #1: Buy groceries
Todo #2: Finish extra lab
Todo #3: Eat banana

'Todo #1: Buy groceries\nTodo #2: Finish extra lab\nTodo #3: Eat banana\n'

In [10]:
mark_complete(1, "bought")

bought

Todo #1: Buy groceries
Todo #2: Finish extra lab
Todo #3: Eat banana

'Todo #1: [green][strike]Buy groceries[/strike][/green]\nTodo #2: Finish extra lab\nTodo #3: Eat banana\n'

In [11]:
create_todos_json = {
    "name": "create_todos",
    "description": "Add new todos from a list of descriptions and return the full list",
    "parameters": {
        "type": "object",
        "properties": {
            "descriptions": {
                'type': 'array',
                'items': {'type': 'string'},
                'title': 'Descriptions'
                }
            },
        "required": ["descriptions"],
        "additionalProperties": False
    }
}

In [12]:
mark_complete_json = {
    "name": "mark_complete",
    "description": "Mark complete the todo at the given position (starting from 1) and return the full list",
    "parameters": {
        'properties': {
            'index': {
                'description': 'The 1-based index of the todo to mark as complete',
                'title': 'Index',
                'type': 'integer'
                },
            'completion_notes': {
                'description': 'Notes about how you completed the todo in rich console markup',
                'title': 'Completion Notes',
                'type': 'string'
                }
            },
        'required': ['index', 'completion_notes'],
        'type': 'object',
        'additionalProperties': False
    }
}

In [13]:
tools = [{"type": "function", "function": create_todos_json},
        {"type": "function", "function": mark_complete_json}]

In [14]:
def handle_tool_calls(tool_calls):
    results = []
    for tool_call in tool_calls:
        tool_name = tool_call.function.name
        arguments = json.loads(tool_call.function.arguments)
        tool = globals().get(tool_name)
        result = tool(**arguments) if tool else {}
        results.append({"role": "tool","content": json.dumps(result),"tool_call_id": tool_call.id})
    return results

In [19]:
def loop(messages):
    done = False
    while not done:
        response = openai.chat.completions.create(model="gemma4", messages=messages, tools=tools, reasoning_effort="none")
        finish_reason = response.choices[0].finish_reason
        if finish_reason=="tool_calls":
            message = response.choices[0].message
            tool_calls = message.tool_calls
            results = handle_tool_calls(tool_calls)
            messages.append(message)
            messages.extend(results)
        else:
            done = True
    show(response.choices[0].message.content)

In [20]:
system_message = """
You are given a problem to solve, by using your todo tools to plan a list of steps, then carrying out each step in turn.
Now use the todo list tools, create a plan, carry out the steps, and reply with the solution.
If any quantity isn't provided in the question, then include a step to come up with a reasonable estimate.
Provide your solution in Rich console markup without code blocks.
Do not ask the user questions or clarification; respond only with the answer after using your tools.
"""
user_message = """"
A train leaves Boston at 2:00 pm traveling 60 mph.
Another train leaves New York at 3:00 pm traveling 80 mph toward Boston.
When do they meet?
"""
messages = [{"role": "system", "content": system_message}, {"role": "user", "content": user_message}]

In [21]:
todos, completed = [], []
loop(messages)

Todo #1: Define the starting positions and times for both trains.
Todo #2: Calculate the head start distance of the first train (Boston to New York).
Todo #3: Determine the relative speed at which the two trains are approaching each other.
Todo #4: Calculate the time taken to meet using the remaining distance and relative speed.
Todo #5: Convert the elapsed time from hours/minutes into a final meeting time.

Boston train (Train A): Starts at 2:00 pm, Speed = 60 mph. New York train (Train B): Starts at 3:00 pm, Speed = 80 
mph. The distance between Boston and New York is not provided; this is a necessary missing piece of information, so
I must make a reasonable estimate for the total distance.

Todo #1: Define the starting positions and times for both trains.
Todo #2: Calculate the head start distance of the first train (Boston to New York).
Todo #3: Determine the relative speed at which the two trains are approaching each other.
Todo #4: Calculate the time taken to meet using the remaining distance and relative speed.
Todo #5: Convert the elapsed time from hours/minutes into a final meeting time.

Since the total distance between Boston and New York is not provided, I will assume a plausible, round number for 
this calculation to demonstrate the method. A common approximation for such problems might use 200 miles or 300 
miles. Given the speeds (60mph and 80mph), **I will estimate the total distance between Boston and New York to be 
300 miles.**\nHead start time: Train B starts 1 hour later (3:00 pm vs 2:00 pm). \nHead start distance for Train A:
$60 \text{ mph} \times 1 \text{ hour} = 60 \text{ miles}$.

Todo #1: Define the starting positions and times for both trains.
Todo #2: Calculate the head start distance of the first train (Boston to New York).
Todo #3: Determine the relative speed at which the two trains are approaching each other.
Todo #4: Calculate the time taken to meet using the remaining distance and relative speed.
Todo #5: Convert the elapsed time from hours/minutes into a final meeting time.

...

Time to Meet (after 3:00 pm) = Remaining Distance / Relative Speed\nTime to Meet = $240 \text{ miles} / 140 \text{ 
mph}\nTime to Meet $\approx 1.714$ hours.

Convert the decimal hours to minutes:\nMinutes = $0.714 \times 60 \text{ minutes/hour} \approx 42.84 \text{ 
minutes}$.\nSo, the time is approximately 1 hour and 43 minutes (rounding up from 0.84 minutes).\nTime to Meet 
$\approx 1$ hour, $42$ minutes, and $50$ seconds ($\frac{24}{14}$ hours).

Todo #1: Define the starting positions and times for both trains.
Todo #2: Calculate the head start distance of the first train (Boston to New York).
Todo #3: Determine the relative speed at which the two trains are approaching each other.
Todo #4: Calculate the time taken to meet using the remaining distance and relative speed.
Todo #5: Convert the elapsed time from hours/minutes into a final meeting time.

The meeting occurs $1$ hour, $42$ minutes, and $50$ seconds after 3:00 pm.

Meeting Time = 3:00 pm + 1 hour, 42 minutes, 50 seconds
Meeting Time $\approx \text{4:43 pm.}$

Todo #1: Define the starting positions and times for both trains.
Todo #2: Calculate the head start distance of the first train (Boston to New York).
Todo #3: Determine the relative speed at which the two trains are approaching each other.
Todo #4: Calculate the time taken to meet using the remaining distance and relative speed.
Todo #5: Convert the elapsed time from hours/minutes into a final meeting time.

The problem requires an assumption for the total distance between Boston and New York, as this crucial piece of 
information was not provided. A reasonable estimate of **300 miles** was used for the calculation.

Based on the calculations:
1.  **Train A (Boston):** Starts at 2:00 pm traveling 60 mph.
2.  **Head Start:** By 3:00 pm, Train A has traveled $60 \text{ miles}$.
3.  **Remaining Distance (at 3:00 pm):** $300 \text{ miles} - 60 \text{ miles} = 240 \text{ miles}$.
4.  **Relative Speed:** $60 \text{ mph} + 80 \text{ mph} = 140 \text{ mph}$.
5.  **Time to Meet (after 3:00 pm):** $\frac{240}{140} = \frac{12}{7}$ hours, or approximately $1.714$ hours 
($\approx 1$ hour, $43$ minutes).
6.  **Meeting Time:** 3:00 pm + 1 hour and about 43 minutes $\approx 4:43$ pm.

Assuming the total distance between Boston and New York is 300 miles, the trains will meet at approximately **4:43 
pm**.

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/exercise.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Exercise</h2>
            <span style="color:#ff7800;">Now try to build an Agent Loop from scratch yourself!<br/>
            Create a new .ipynb and make one from first principles, referring back to this as needed.<br/>
            It's one of the few times that I recommend typing from scratch - it's a very satisfying result.
            </span>
        </td>
    </tr>
</table>